# 三角代理对话（莎士比亚角色扮演）

## 练习目标（理念）

用 **三个不同模型** 分别扮演莎士比亚戏剧人物，形成三角对话：

- **GPT（`gpt-4o-mini`）** → 哈姆雷特（Hamlet）
- **本地 Ollama（`llama3.2`）** → 福斯塔夫（Falstaff）
- **Gemini（`gemini-2.0-flash`）** → 伊阿古（Iago）

后半段还会演示 **双模型对聊**（孟买 tapori vs 哈里亚纳 Jaat）。

## 和本课第 2 周的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| 多模型 / 多供应商 | OpenAI + Ollama + Google Generative AI |
| `system` 角色提示 | `gpt_system` / `llama_system` / `gemini_system` |
| 对话历史 `messages` | 用三个列表交错拼成发给每个模型的上下文 |
| 本地开源模型 | `ollama.chat(model=llama_model, ...)` |

## 怎么跑

1. 从上到下依次运行单元格（Shift+Enter）
2. `.env` 准备 `OPENAI_API_KEY`、`GOOGLE_API_KEY`；本机需 `ollama pull llama3.2`
3. 先跑三角对话循环，再跑 GPT ↔ Llama 双人对话


### 练习要点

1. 搭一条「三路」对话：把 Gemini 也拉进谈话。
2. 把其中一个云端模型换成用 **Ollama** 跑的开源模型（本笔记本用 `llama3.2`）。


In [ ]:
# ========== 导入：OpenAI / 环境变量 / 展示 / Ollama ==========

# 标准库 os：读环境变量（Environment Variables），例如 API Key
import os
# load_dotenv：把 .env 里的密钥读进进程环境，避免把密钥写进代码
from dotenv import load_dotenv
# OpenAI 客户端：调用云端 Chat Completions（本练习给 Hamlet 用）
from openai import OpenAI
# IPython 展示工具：Markdown / display / update_display（本格导入后备用）
from IPython.display import Markdown, display, update_display
# ollama Python SDK：调本地 Ollama 模型（本练习给 Falstaff 用）
import ollama


In [ ]:
# ========== 导入 Google Generative AI SDK ==========

# google.generativeai：Gemini 官方 Python 包（后面 configure + GenerativeModel）
import google.generativeai


In [ ]:
# ========== 加载环境变量，并做密钥前缀自检（调试用）==========

# override=True：.env 里的值覆盖进程里已有同名环境变量
load_dotenv(override=True)
# 从环境读取 OpenAI 密钥；不存在则为 None
openai_api_key = os.getenv('OPENAI_API_KEY')
# 从环境读取 Google / Gemini 密钥
google_api_key = os.getenv('GOOGLE_API_KEY')

# 有密钥就打印前 8 位前缀，方便确认「读到了」又不暴露完整密钥
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

# 对 Google 密钥做同样的存在性检查
if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:8]}")
else:
    print("Google API Key not set")


In [ ]:
# ========== 创建 OpenAI 客户端 ==========

# 默认从环境变量 OPENAI_API_KEY 取密钥；后面 openai.chat.completions.create 都走它
openai = OpenAI()


In [ ]:
# ========== 配置 Google Generative AI ==========

# configure()：按 SDK 默认方式读 GOOGLE_API_KEY 等环境变量，初始化 Gemini 调用环境
google.generativeai.configure()


In [ ]:
# ========== 模型名 + 三角角色 system 提示 + 开场白列表 ==========

# OpenAI 侧模型 id：便宜小模型，扮演 Hamlet
gpt_model = "gpt-4o-mini"
# 本地 Ollama 模型名：需事先 ollama pull；扮演 Falstaff
llama_model = "llama3.2"
# Gemini 模型 id：Flash 偏快；扮演 Iago
gemini_model = 'gemini-2.0-flash'

# 发给 GPT 的 system：要求用莎士比亚《哈姆雷特》口吻回应（英文 prompt 保持原样，改译会改行为）
gpt_system = "You are playing part of Hamlet. he is philosopher, probes Iago with a mixture of suspicion\
and intellectual curiosity, seeking to unearth the origins of his deceit.\
Is malice born of scorn, envy, or some deeper void? Hamlet’s introspective nature\
drives him to question whether Iago’s actions reveal a truth about humanity itself.\
You will respond as Shakespear's Hamlet will do."

# 发给 Llama 的 system：福斯塔夫式插科打诨，冲淡哈姆雷特的忧郁
llama_system = "You are acting part of Falstaff who attempts to lighten the mood with his jokes and observations,\
potentially clashing with Hamlet's melancholic nature.You respond as Shakespear's Falstaff do."

# 发给 Gemini 的 system：伊阿古式暗中操纵双方
gemini_system = "You are acting part of Iago, subtly trying to manipulate both Hamlet and Falstaff\
to his own advantage, testing their weaknesses and exploiting their flaws. You respond like Iago"

# 三个列表各存一方「自己说过的话」；开场白先各放一句，后面循环再 append
gpt_messages = ["Hi there"]
llama_messages = ["Hi"]
gemini_messages = ["Hello"]


In [ ]:
# ========== 调用 GPT（Hamlet）：拼三角历史后 chat.completions ==========

def call_gpt():
    # 先放 system，定哈姆雷特人设
    messages = [{"role": "system", "content": gpt_system}]
    # zip 三份历史：对 GPT 而言，自己的话是 assistant，另两方是 user
    for gpt, claude, gemini in zip(gpt_messages, llama_messages, gemini_messages):
        messages.append({"role": "assistant", "content": gpt})
        messages.append({"role": "user", "content": claude})
        messages.append({"role": "user", "content": gemini})
    # 非流式一次拿完整段回复
    completion = openai.chat.completions.create(
        model=gpt_model,
        messages=messages
    )
    # 取出第一条 choice 的文本内容返回
    return completion.choices[0].message.content


In [ ]:
# ========== 调用本地 Llama（Falstaff）：ollama.chat ==========

def call_llama():
    # Ollama 侧 messages 从空列表开始拼（本实现未单独塞 system，保持原逻辑）
    messages = []
    # zip 三份历史：Llama 自己的话是 assistant，Hamlet/Iago 是 user
    for gpt, llama, gemini in zip(gpt_messages, llama_messages, gemini_messages):
        messages.append({"role": "user", "content": gpt})
        messages.append({"role": "assistant", "content": llama})
        messages.append({"role": "user", "content": gemini})
    # 再追加最新一句 Hamlet，当作「轮到你答」的触发
    messages.append({"role": "user", "content": gpt_messages[-1]})
    # 调本地 Ollama；model 字符串必须和本机已 pull 的名字一致
    response = ollama.chat(model=llama_model, messages=messages)

   
    # Ollama 返回 dict：message.content 是助手文本
    return response['message']['content']


In [ ]:
# ========== 调用 Gemini（Iago）：GenerativeModel + system_instruction ==========

# 将 google.generativeai 导入为 genai（注释提示；本格实际仍用 google.generativeai）

# Make sure you configure the API 密钥 first:
# genai.configure(api_key="YOUR_API_KEY")

def call_gemini():
    # 注意：这里同名局部变量会盖住模块级 gemini_messages（原笔记本逻辑如此，教学注释不改行为）
    gemini_messages = []
    
    # 格式化 Gemini 的历史记录（Gemini API 用 role=user/model + parts）
    for gpt, llama, gemini_message in zip(gpt_messages, llama_messages, gemini_messages):
        gemini_messages.append({"role": "user", "parts": [gpt]})         # Hamlet speaks
        gemini_messages.append({"role": "model", "parts": [llama]})      # Falstaff responds
        gemini_messages.append({"role": "model", "parts": [gemini_message]})  # Iago responds

    # 如果需要，添加最新的用户输入（可选）：用最新一句 Falstaff 当 user 触发
    gemini_messages.append({"role": "user", "parts": [llama_messages[-1]]})

    # 初始化 the model with the correct system instruction（伊阿古人设）
    gemini = google.generativeai.GenerativeModel(
        # model_name='gemini-1.5-flash', # 或 'gemini-pro'
        model_name = gemini_model,
        system_instruction=gemini_system
    )

    # generate_content：把拼好的历史交给 Gemini，取文本
    response = gemini.generate_content(gemini_messages)
    return response.text
# 打印（响应.文本）


In [ ]:
# ========== 三角对话主循环：各说 3 轮并打印 ==========

# 重新初始化三方开场白（与上面定义一致，便于单独重跑本格）
gpt_messages = ["Hi there"]
llama_messages = ["Hi"]
gemini_messages = ["Hello"]

# 先打印三方开场
print(f"Hamlet:\n{gpt_messages[0]}\n")
print(f"Falstaff:\n{llama_messages[0]}\n")
print(f"Iago:\n{gemini_messages[0]}\n")

# 固定跑 3 轮：GPT → Llama → Gemini
for i in range(3):
    # Hamlet 发言，并写入自己的历史
    gpt_next = call_gpt()
    print(f"GPT:\n{gpt_next}\n")
    gpt_messages.append(gpt_next)
    
    # Falstaff 发言，并写入自己的历史
    llama_next = call_llama()
    print(f"Llama:\n{llama_next}\n")
    llama_messages.append(llama_next)

    # Iago 发言；注意：原逻辑把 Gemini 回复 append 进了 llama_messages（保持不改）
    gemini_next = call_gemini()
    print(f"Gemini:\n{gemini_next}\n")
    llama_messages.append(gemini_next)


## gpt-4o-mini 与 llama3.2 之间的双人对聊

下面换成 **两方对话**：不再用莎士比亚三角，而用「孟买 tapori」与「哈里亚纳 Jaat」两个人设对聊。


In [ ]:
# ========== 双人对话：模型名 + 人设 system + 开场白 ==========

# 让我们在 GPT-4o-mini 和 Claude-3-haiku 之间进行对话（注释沿用原文；实际第二方是 Ollama llama3.2）
# 我们使用廉价版本的模型，因此成本将是最低的

# 云端小模型
gpt_model = "gpt-4o-mini"
# 本地开源模型
llama_model = "llama3.2"

# GPT 人设：乐观的孟买 tapori（英文 prompt 不翻译）
gpt_system = "You are a tapori from mumbai who is very optimistic; \
you alway look at the brighter part of the situation and you always ready to take act to win way."

# Llama 人设：爱用印地语诗歌找共同点的 Jaat
llama_system = "You are a Jaat from Haryana. You try to express with hindi poems \
to agree with other person and or find common ground. If the other person is optimistic, \
you respond in poetic way and keep chatting."

# 双方各一句开场
gpt_messages = ["Hi there"]
llama_messages = ["Hi"]


In [ ]:
# ========== 双人版 call_gpt：只 zip GPT 与 Llama 两份历史 ==========

def call_gpt():
    # system 定 tapori 人设
    messages = [{"role": "system", "content": gpt_system}]
    # 自己的话 assistant，对方的话 user
    for gpt, llama in zip(gpt_messages, llama_messages):
        messages.append({"role": "assistant", "content": gpt})
        messages.append({"role": "user", "content": llama})
    # 调用 OpenAI Chat Completions
    completion = openai.chat.completions.create(
        model=gpt_model,
        messages=messages
    )
    return completion.choices[0].message.content


In [ ]:
# ========== 双人版 call_llama：拼历史后 ollama.chat ==========

def call_llama():
    # 从空 messages 开始
    messages = []
    # zip 双方历史：GPT 为 user，Llama 为 assistant
    for gpt, llama_message in zip(gpt_messages, llama_messages):
        messages.append({"role": "user", "content": gpt})
        messages.append({"role": "assistant", "content": llama_message})
    # 追加最新 GPT 句，触发本轮回复
    messages.append({"role": "user", "content": gpt_messages[-1]})
    # 调本地 Ollama
    response = ollama.chat(model=llama_model, messages=messages)

   
    return response['message']['content']


In [ ]:
# ========== 双人对话主循环：交替 3 轮 ==========

# 重置开场白，便于单独重跑
gpt_messages = ["Hi there"]
llama_messages = ["Hi"]

# 打印开场
print(f"GPT:\n{gpt_messages[0]}\n")
print(f"Llama:\n{llama_messages[0]}\n")

# 交替：GPT 答 → 写入 → Llama 答 → 写入
for i in range(3):
    gpt_next = call_gpt()
    print(f"GPT:\n{gpt_next}\n")
    gpt_messages.append(gpt_next)
    
    llama_next = call_llama()
    print(f"Llama:\n{llama_next}\n")
    llama_messages.append(llama_next)
